# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIRˆ2 clinical tabular dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

This dataset holds clinical, pathological, anatomical, and molecular information for 77 cancer survivors with second primary colorectal cancer, supporting biomarker and anatomical distribution analyses.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access metadata object and print description
metadata = dataset.metadata  # DatasetMetadata object
print(f"{metadata.name}: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Published: {metadata.datePublished}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

For Croissant datasets, every entity is referenced by its `@id`. Let's enumerate available record sets, fields and columns, referencing their IDs accordingly.

In [ ]:
record_sets = metadata.recordSets
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}\n  Name: {rs.name}\n  Description: {rs.description}\n  Fields:")
    for field in rs.fields:
        print(f"    Field @id: {field.id}\n      Name: {field.name}\n      DataType: {field.dataType}\n      Description: {field.description}")
    print("  Columns:")
    for col in rs.columns:
        print(f"    Column @id: {col.id}\n      Name: {col.name}\n      Description: {col.description}\n      DataType: {col.dataType}")
    print("\n---\n")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.
Here we demonstrate loading all available record sets as pandas DataFrames, referencing by `@id`.

In [ ]:
# Extract data from each record set
# List all recordSet @ids
record_set_ids = [rs.id for rs in metadata.recordSets]
dataframes = {}

for record_set_id in record_set_ids:
    # Load all records as dictionaries
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for record set {record_set_id} with shape {df.shape}")

# Example: Show columns for the first record set
if record_set_ids:
    print(f"Columns for {record_set_ids[0]}:")
    print(dataframes[record_set_ids[0]].columns.tolist())
    print(dataframes[record_set_ids[0]].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Below, we choose a numeric field (using its `@id`) and perform filtering, normalization, and grouping.

In [ ]:
# Choose a record set and a numeric field by their @ids
# Let's select the first record set as an example
example_record_set_id = record_set_ids[0]
example_df = dataframes[example_record_set_id]

# Identify numeric columns (fields) by their @id
numeric_field_id = None
for field in metadata.recordSets[0].fields:
    if field.dataType in ['Integer', 'Float', 'Number']:
        numeric_field_id = field.id
        break

if numeric_field_id is not None:
    numeric_field_name = numeric_field_id
    # Filtering: e.g., values > threshold
    threshold = 50  # Example threshold
    if numeric_field_name in example_df.columns:
        filtered_df = example_df[example_df[numeric_field_name] > threshold]
        print(f"Filtered records with {numeric_field_name} > {threshold}:")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_name}_normalized"] = (
            filtered_df[numeric_field_name] - filtered_df[numeric_field_name].mean()
        ) / filtered_df[numeric_field_name].std()
        print(f"Normalized {numeric_field_name} for filtered records:")
        print(filtered_df[[numeric_field_name, f"{numeric_field_name}_normalized"]].head())
        
        # Group by another field (@id)
        group_field_id = None
        for field in metadata.recordSets[0].fields:
            if field.dataType == 'Text' and field.id != numeric_field_id:
                group_field_id = field.id
                break
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_name].mean().reset_index()
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
else:
    print("No numeric field found in the first record set for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we plot the distribution of the chosen numeric field and show a basic group bar plot.

In [ ]:
import matplotlib.pyplot as plt

if numeric_field_id is not None and numeric_field_name in example_df.columns:
    plt.figure(figsize=(7,4))
    example_df[numeric_field_name].hist(bins=10)
    plt.title(f"Distribution of {numeric_field_name}")
    plt.xlabel(numeric_field_name)
    plt.ylabel("Count")
    plt.show()

    # Plot normalized values
    if f"{numeric_field_name}_normalized" in filtered_df.columns:
        plt.figure(figsize=(7,4))
        filtered_df[f"{numeric_field_name}_normalized"].hist(bins=10)
        plt.title(f"Normalized Distribution of {numeric_field_name}")
        plt.xlabel(f"{numeric_field_name}_normalized")
        plt.ylabel("Count")
        plt.show()

    # Grouped bar plot
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_means = filtered_df.groupby(group_field_id)[numeric_field_name].mean()
        grouped_means.plot(kind='bar', figsize=(8,4))
        plt.title(f"Mean of {numeric_field_name} by {group_field_id}")
        plt.ylabel(f"Mean {numeric_field_name}")
        plt.xlabel(group_field_id)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIRˆ2 dataset provides a rich set of clinical, pathological, and molecular data for second primary colorectal cancer survivors.
- Record sets, fields, and columns can be programmatically accessed and referenced using their `@id`.
- Simple EDA and visualization show the dataset is well-structured for clinical analytics, with demographic and molecular fields available.
- The dataset is suitable for biomarker stratification, anatomical site analysis, and treatment outcome studies within the described survivor population.
- The exploratory analysis demonstrates how to filter and normalize numeric fields (by `@id`), and group by clinical text fields, supporting replicable FAIR workflows.